# SQL assistant: run the tests and the accuracy benchmark

Runs the project in Google Colab, with nothing to install on your computer. Run the cells in order with Shift+Enter.

## 1. Set up

Downloads the project, installs its packages and runs the tests. The tests need no model.

In [ ]:
!git clone -q https://github.com/emmanuelkiprop448-sudo/llm-sql-assistant.git
%cd llm-sql-assistant
!pip -q install -r requirements.txt
!pytest -q

## 2. Connect a model

The benchmark needs a language model. A free option:

1. Sign in at [aistudio.google.com](https://aistudio.google.com) with your Gmail account and create an API key.
2. In Colab, open the **key icon (Secrets)** in the left sidebar.
3. Add a secret named `LLM_API_KEY`, paste the key as its value, and switch on **Notebook access**.

The key stays in your Colab secrets and is never saved in the notebook. `LLM_MODEL` below can be any model Google AI Studio offers on your plan.

In [ ]:
import os
from google.colab import userdata

os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
os.environ["LLM_MODEL"] = "gemini-3.5-flash"
os.environ["LLM_BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"

## 3. Ask a question

The model writes the SQL, the guardrails check it, and the query runs on a read-only connection.

In [ ]:
import pandas as pd
from sql_assistant import db, guardrails, llm

QUESTION = "Which five countries have the most customers?"

path = db.ensure_database()
reply = llm.generate_sql(QUESTION, db.describe_schema(path), llm.load_config())
sql = guardrails.extract_sql(reply)
print(sql, "\n")

conn = db.connect_guarded(path)
check = guardrails.validate(sql, conn)
for c in check.checks:
    print("PASS" if c.passed else "FAIL", c.name, c.detail)
if check.ok:
    result = guardrails.run_query(conn, check.sql)
    display(pd.DataFrame(result.rows, columns=result.columns))

## 4. Run the benchmark

Asks the model all 12 benchmark questions and scores each answer against the reference result. The pause between questions keeps within free-plan rate limits; it takes a few minutes.

In [ ]:
!python -m sql_assistant.evaluate --pause 6

from IPython.display import Markdown, display
display(Markdown(open("eval/results/summary.md").read()))

## 5. Save the results

Downloads `summary.md` and `results.csv`. To publish them, upload the `eval/results/` folder to the GitHub repository and add the model name and accuracy to the README.

In [ ]:
!zip -qr eval-results.zip eval/results
from google.colab import files
files.download("eval-results.zip")